# KAF-6 — Poison Pill / Dead-Letter

**Break -> Detect -> Fix -> Prove.** One corrupt, unparseable message in a partition can stall a
naive consumer (it keeps failing on the *same* offset) or crash the job. The production cure is the
**dead-letter** pattern: split the stream, quarantine the bad records, keep moving.

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers, `kafka:9092` for Spark) --
browse topics live in **kafka-ui** at http://localhost:8080. **Laptop-safe:** a small bounded batch
(~500 good + a few poison messages), read with `.trigger(availableNow=True)` so the stream consumes
everything and **stops on its own**. Teardown deletes the topic + Iceberg tables (`make clean` also
clears local state).

See the companion writeup in [`README.md`](./README.md).

In [ ]:
import json

from kafka import KafkaProducer
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType

from common.spark_session import spark, display_df
from common.kafka_helpers import ensure_topic, produce_events, delete_topic, BOOTSTRAP, SPARK_BOOTSTRAP

TOPIC          = "kaf6_orders"
MAIN_TABLE     = "iceberg_catalog.default.kaf6_orders_clean"
DLQ_TABLE      = "iceberg_catalog.default.kaf6_orders_deadletter"
MAIN_CKPT      = ".tmp/checkpoint_kaf6_main"
DLQ_CKPT       = ".tmp/checkpoint_kaf6_dlq"

N_GOOD = 500   # good JSON events
print("producers ->", BOOTSTRAP, "| Spark reads <-", SPARK_BOOTSTRAP, "| kafka-ui:", "http://localhost:8080")

## Step 0 -- a fresh topic and clean tables

Recreate the topic (a single partition is enough to show one bad offset blocking progress) and drop
any tables / checkpoints from a previous run so the counts are deterministic.

In [ ]:
import shutil

ensure_topic(TOPIC, num_partitions=1, recreate=True)

# Start clean so produced == main + dlq holds exactly.
spark.sql(f"DROP TABLE IF EXISTS {MAIN_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {DLQ_TABLE}")
for _ckpt in (MAIN_CKPT, DLQ_CKPT):
    shutil.rmtree(_ckpt, ignore_errors=True)

print(f"topic {TOPIC} ready (1 partition); old tables/checkpoints cleared")

## 1. Break it -- inject poison among good events

`produce_events` JSON-encodes whatever `value_fn` returns, so it can only ever emit *valid* JSON.
To inject genuinely malformed bytes we drop to a raw `kafka-python` producer and `.send(...)` the
poison directly. We mix two kinds of poison in with the good events:

- `b"NOT-JSON-this-is-a-poison-pill"` -- not JSON at all,
- a JSON object missing the required `id` field -- right format, **wrong schema**.

In [ ]:
# 500 good JSON order events via the shared helper.
produce_events(TOPIC, N_GOOD, value_fn=lambda i: {"id": i, "amount": round(i * 1.5, 2), "status": "ok"})

# A raw producer that passes bytes straight through (and JSON-encodes dicts) so we can inject poison.
raw = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode("utf-8"),
)
POISON = [
    b"NOT-JSON-this-is-a-poison-pill",              # unparseable bytes
    b"{ broken json, no quotes }",                  # looks JSON-ish, still unparseable
    {"oops": "wrong schema", "amount": 9.9},        # valid JSON, but no required `id`
]
for p in POISON:
    raw.send(TOPIC, p)
raw.flush()
raw.close()

N_POISON   = len(POISON)
N_TOTAL    = N_GOOD + N_POISON
print(f"produced {N_GOOD} good + {N_POISON} poison = {N_TOTAL} total messages to {TOPIC}")

### Read it -- `from_json` in PERMISSIVE mode returns all-NULL for bad rows

Spark's default JSON parse mode is **PERMISSIVE**: an unparseable row does **not** raise -- the
parsed struct comes back **all-NULL**. So a naive "parse everything and write it" pipeline silently
lands NULL rows and the corruption goes unnoticed. (We read the bounded topic with a batch
`spark.read` here to inspect; the streaming writes come later.)

In [ ]:
schema = StructType([
    StructField("id",     LongType(),   True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
])

raw_df = (spark.read.format("kafka")
          .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
          .option("subscribe", TOPIC)
          .option("startingOffsets", "earliest")
          .load()
          .select(F.col("value").cast("string").alias("value"),
                  "topic", "partition", "offset", "timestamp"))

parsed_df = raw_df.withColumn("parsed", F.from_json("value", schema))   # PERMISSIVE (default)

print("Poison rows parse to an all-NULL struct (value is non-empty, but `parsed` is NULL):")
(parsed_df.filter(F.col("parsed.id").isNull())
          .select("value", "parsed")
          .show(10, truncate=60))

### The other failure mode -- `FAILFAST` takes down the whole batch

Opt into `mode='FAILFAST'` and instead of silent NULLs, a **single** bad row makes the **entire
batch throw**. So the naive choice is lose-lose: PERMISSIVE writes garbage silently; FAILFAST
crashes on one poison message (and a bare decode-and-loop consumer would get stuck re-reading that
offset forever -- the partition is *blocked*).

In [ ]:
# Force evaluation so the FAILFAST parser actually runs over the poison rows.
try:
    (raw_df.select(F.from_json("value", schema, {"mode": "FAILFAST"}).alias("p"))
           .filter(F.col("p.id").isNotNull())   # an action forces the parse
           .count())
    print("(no error -- unexpected)")
except Exception as e:                          # noqa: BLE001 -- demonstrating the crash
    msg = str(e).splitlines()[0]
    print("FAILFAST batch FAILED on a poison row (this is the crash a naive job hits):")
    print("  ", msg[:160])

## 2. Detect it -- required field NULL while value is non-empty = poison

The signature of a poison pill: the parsed struct (or a **required** field) is NULL while the raw
`value` is non-empty. Anchoring on a required field (`id`) distinguishes true poison from a
legitimately empty payload. Count them -- this is the number that must end up in the DLQ.

In [ ]:
is_poison = F.col("parsed.id").isNull() & F.col("value").isNotNull() & (F.length("value") > 0)

detected_poison = parsed_df.filter(is_poison).count()
detected_good   = parsed_df.filter(~is_poison).count()
print(f"detected poison rows : {detected_poison}")
print(f"detected good rows   : {detected_good}")
print(f"total                : {detected_good + detected_poison}  (== {N_TOTAL} produced)")

## 3. Fix it -- the dead-letter pattern (split the stream, write both sides)

Split by parse outcome and write **both** sides with `.trigger(availableNow=True)` (bounded: each
stream consumes all available data and **stops on its own**):

- `parsed.id IS NOT NULL` -> the **main** Iceberg table (flattened, typed columns),
- `parsed.id IS NULL`     -> a **dead-letter (DLQ)** table that keeps the raw `value` plus
  `topic`, `partition`, `offset`, `timestamp` for forensics / replay.

Each write has its **own checkpoint**, so the poison becomes a DLQ row -- not an outage. The
partition is never blocked. (Two `availableNow` writes is the simplest Connect-safe form of the
split; `foreachBatch` with two writes inside one batch is the equivalent single-stream variant.)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {MAIN_TABLE} (
        id      BIGINT,
        amount  DOUBLE,
        status  STRING
    ) USING iceberg TBLPROPERTIES ('format-version' = '2')
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DLQ_TABLE} (
        value      STRING,
        topic      STRING,
        partition  INT,
        offset     BIGINT,
        timestamp  TIMESTAMP
    ) USING iceberg TBLPROPERTIES ('format-version' = '2')
""")

print("main + dead-letter tables ready")

In [ ]:
# Streaming source over the same bounded topic.
stream = (spark.readStream.format("kafka")
          .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
          .option("subscribe", TOPIC)
          .option("startingOffsets", "earliest")
          .load()
          .select(F.col("value").cast("string").alias("value"),
                  "topic", "partition", "offset", "timestamp")
          .withColumn("parsed", F.from_json("value", schema)))

poison_flag = F.col("parsed.id").isNull()

# (A) Good rows -> main table.
clean_df = stream.filter(~poison_flag).select("parsed.id", "parsed.amount", "parsed.status")
q_main = (clean_df.writeStream.format("iceberg").outputMode("append")
          .option("checkpointLocation", MAIN_CKPT)
          .trigger(availableNow=True)
          .toTable(MAIN_TABLE))

# (B) Poison rows -> dead-letter table (keep raw value + Kafka coordinates).
dlq_df = stream.filter(poison_flag).select("value", "topic", "partition", "offset", "timestamp")
q_dlq = (dlq_df.writeStream.format("iceberg").outputMode("append")
         .option("checkpointLocation", DLQ_CKPT)
         .trigger(availableNow=True)
         .toTable(DLQ_TABLE))

# Bounded: each stream drains all available data, then terminates.
q_main.awaitTermination()
q_dlq.awaitTermination()
print("both availableNow streams completed -- pipeline did NOT stall on the poison message")

## 4. Prove it -- good + poison == total, and the job finished

The arithmetic that proves both **correctness** (nothing lost, nothing silently corrupted) and
**liveness** (the poison didn't block the partition):

```
main_count (parsed OK)  +  dlq_count (poison)  ==  total produced
```

In [ ]:
main_count = spark.table(MAIN_TABLE).count()
dlq_count  = spark.table(DLQ_TABLE).count()

print(f"main table (clean) : {main_count}")
print(f"dead-letter (DLQ)  : {dlq_count}")
print(f"sum                : {main_count + dlq_count}")
print(f"produced           : {N_TOTAL}")
print(f"RECONCILES         : {main_count + dlq_count == N_TOTAL}  "
      f"(good={main_count == N_GOOD}, poison={dlq_count == N_POISON})")

print("\nThe quarantined poison messages (raw bytes kept for replay):")
display_df(spark.table(DLQ_TABLE).select("value", "partition", "offset"), limit=10)

## Takeaways & "in real production..."

- **Never let one bad message block a partition.** Decouple progress from parse-success -- isolate
  unparseable records to a **DLQ** and commit past them.
- **Don't trust PERMISSIVE silence.** All-NULL `from_json` structs are corruption hiding in plain
  sight -- explicitly detect `required_field IS NULL AND value IS NOT NULL` and route it.
- **Alert on the DLQ rate.** A non-zero / rising dead-letter rate is the early warning that an
  upstream producer went bad -- page on it.
- **Validate schema upstream** (schema registry + compatibility checks, or producer-side
  validation) -- the DLQ is the safety net, not the primary defense.
- **Keep raw bytes + Kafka coordinates** in the DLQ so you can diagnose and **replay** once the
  upstream bug is fixed.

## Teardown

In [ ]:
delete_topic(TOPIC)
spark.sql(f"DROP TABLE IF EXISTS {MAIN_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {DLQ_TABLE}")
for _ckpt in (MAIN_CKPT, DLQ_CKPT):
    shutil.rmtree(_ckpt, ignore_errors=True)
print("Deleted topic + main/DLQ tables + checkpoints. `make clean` also clears any local state.")